In [5]:
import torch
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} is available.")
else:
    print("No GPU available. Training will run on CPU.")

GPU: Tesla T4 is available.


# NetGuard Benchmark on Colab

In [ ]:
import sys
import os
import subprocess

# Install dependencies
!pip install -q torch pytorch-lightning scikit-learn pandas numpy plotly shap xgboost

# Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Clone and setup
!cd /content && (git clone https://github.com/20Saurabh/Network-Anomaly-Detection.git 2>/dev/null || (cd Network-Anomaly-Detection && git pull))

if os.path.exists('/content/Network-Anomaly-Detection/backend'):
    os.chdir('/content/Network-Anomaly-Detection/backend')
    sys.path.insert(0, '/content/Network-Anomaly-Detection/backend')

    # Set Kaggle API token
    os.environ['KAGGLE_API_TOKEN'] = 'KGAT_eda598a3dd5fe9e00d7041a4cb90d0'

    # Download datasets if not present
    if not os.path.exists('data/raw'):
        print("Downloading datasets...")
        download_result = subprocess.run(['python', 'download_all_datasets.py'])
        if download_result.returncode != 0:
            print("ERROR downloading datasets")
else:
    print("Failed to clone repository. Please check the GitHub URL.")
    # Perhaps exit or handle

# Run benchmark one model at a time
models = ["bilstm_attention", "cnn_lstm", "contrastive_ssl", "ensemble_stacking", "vanilla_ae", "cnn1d", "ft_transformer", "isolation_forest", "vae"]  # Skipped vanilla_ae due to error
for model in models:
    print(f"Starting benchmark for {model}...")
    result = subprocess.run(
        f'python -u run_experiments.py --dataset synthetic --epochs 5 --runs 1 --max-samples 5000 --models {model}',
        shell=True, capture_output=True, text=True
    )
    print(result.stdout)
    print(f"Return code for {model}: {result.returncode}")
    if result.returncode != 0:
        print(f"ERROR for {model}: {result.stderr}")
        break  # Stop at first failure

# Check results
print("Checking results directory...")
!ls -la results/ 2>/dev/null || echo "No results directory"

# Save results to Drive
!mkdir -p /content/drive/MyDrive/NetGuard_Results
!cp -r results/* /content/drive/MyDrive/NetGuard_Results/ 2>/dev/null || true

print("✅ Benchmark completed! Check /content/drive/MyDrive/NetGuard_Results for output")

Mounted at /content/drive
remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 7 (delta 5), reused 7 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (7/7), 2.22 KiB | 758.00 KiB/s, done.
From https://github.com/20Saurabh/Network-Anomaly-Detection
   439c92e..86c941f  main       -> origin/main
Updating 439c92e..86c941f
Fast-forward
 backend/explainability/shap_analysis.py |  46 ++++++++---
 backend/run_experiments.py              |   3 +-
 colab.ipynb                             | 132 +++++++++++++++-----------------
 3 files changed, 98 insertions(+), 83 deletions(-)
Starting benchmark for vanilla_ae...
[CONFIG] Using GPU: Tesla T4
[CONFIG] Project root: /content/Network-Anomaly-Detection/backend
[CONFIG] Results dir:  /content/Network-Anomaly-Detection/backend/results

  NETWORK ANOMALY DETECTION BENCHMARK v2.0
  Next-Generation Research-Grade System
  Device: cuda
  Dataset: synthet

## Generate Report

In [24]:
import json
import pandas as pd
import plotly.express as px
import os

# Load results
results_path = '/content/drive/MyDrive/NetGuard_Results/metrics/all_results.json'
if os.path.exists(results_path):
    with open(results_path, 'r') as f:
        results = json.load(f)
    
    # Convert to DataFrame (assuming results is a dict with model keys)
    df = pd.DataFrame.from_dict(results, orient='index').reset_index().rename(columns={'index': 'model'})
    
    # Display summary
    print("Benchmark Results Summary:")
    print(df.describe())
    
    # Plot performance comparison
    fig = px.bar(df, x='model', y='accuracy', title='Model Accuracy Comparison')
    fig.show()
    
    # Save report
    df.to_csv('/content/drive/MyDrive/NetGuard_Results/benchmark_report.csv', index=False)
    print("Report saved to /content/drive/MyDrive/NetGuard_Results/benchmark_report.csv")
else:
    print("Results not found. Run the benchmark first.")

Benchmark Results Summary:
       num_test_samples  num_classes  accuracy  precision    recall  f1_score  \
count               8.0          8.0  9.000000        9.0  9.000000  9.000000   
mean              750.0          2.0  0.997333        1.0  0.973333  0.985816   
std                 0.0          0.0  0.005292        0.0  0.052915  0.028146   
min               750.0          2.0  0.988000        1.0  0.880000  0.936170   
25%               750.0          2.0  1.000000        1.0  1.000000  1.000000   
50%               750.0          2.0  1.000000        1.0  1.000000  1.000000   
75%               750.0          2.0  1.000000        1.0  1.000000  1.000000   
max               750.0          2.0  1.000000        1.0  1.000000  1.000000   

       f1_macro  f1_weighted       auc_roc        auc_pr  detection_rate  \
count  9.000000     8.000000  9.000000e+00  9.000000e+00        8.000000   
mean   0.992172     0.996914  1.000000e+00  1.000000e+00        0.970000   
std    0.015533

Report saved to /content/drive/MyDrive/NetGuard_Results/benchmark_report.csv
